# 层叠、优先级与继承

学习目标：能按层叠顺序追踪生效声明，解释继承与重置，并用层组织可预测的覆盖。

前置知识：CSS 选择器、声明、class/id、父子关系；会在开发者工具中查看 Styles。

适用范围：同一普通文档树中的 CSS 层叠；不包含 Shadow DOM 的封装上下文比较。@layer 与 revert-layer 需要相应浏览器支持，复杂 @scope 在第19章展开。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/03-cascade-and-inheritance/。

1. [index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css)：声明顺序、优先级、继承与来源回退。
2. [layers.html](scripts/03-cascade-and-inheritance/layers.html)、[layers.css](scripts/03-cascade-and-inheritance/layers.css)：层顺序、重要性反转与层回退。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 先看同一属性的竞争

层叠（cascade）为每个元素的每个属性分别选择声明。先过滤不匹配的选择器、未满足的条件和无效声明，再按层叠顺序比较。

本例都属于作者来源、普通声明、没有显式层，选择器也相同，因此最后出现的 color 获胜；font-size 没有竞争，独立保留。外部样式表和内部样式表都属于作者来源，不存在“内部必然高于外部”的固定等级。

下面先用声明顺序建立直觉；来源、重要性和层这些更早的比较阶段不同，就不能直接套用“后写覆盖前写”。

```html
<p class="order-demo">同一段文字的颜色与字号</p>
```

```css
.order-demo { color: teal; font-size: 20px; }
.order-demo { color: navy; }
/* 检查：20px 字号仍保留；只在本例其他比较条件相同后，后面的 navy 胜出。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-1)

## 2 来源、重要性与内联样式

来源包括浏览器默认样式（user agent）、用户样式（user）和页面作者样式（author）。用户样式是浏览器侧设置或用户样式机制提供的规则，不是访问者在页面编辑框里写的任意内容。

普通声明的来源优先次序从低到高是：浏览器、用户、作者。重要声明的来源次序反转：作者 !important、用户 !important、浏览器 !important。!important 是声明末尾的标记，不是属性值，也不增加选择器优先级。

完整来源顺序还包含动画和过渡：普通作者声明之上是关键帧动画值，再往上是三种重要声明，正在生效的过渡值位于最高阶段。本章没有动画，不把“!important 永远最高”作为规则。

同一来源、同一封装上下文中，style 属性直接附着的声明优先于同等重要性的样式规则；普通内联仍会输给作者的重要声明。下面只示范这个条件，不借助复杂动画。

```html
<p id="inline-normal" class="importance-demo" style="color: purple;">普通内联颜色</p>
<p id="inline-important" class="importance-demo" style="color: purple !important;">重要内联颜色</p>
```

```css
.importance-demo { color: navy !important; }
/* 第一段为 navy：作者重要规则覆盖普通内联；第二段为 purple：重要内联优先。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-2)

## 3 选择器优先级怎样比较

选择器优先级（specificity）只在来源、重要性、附着方式和层等更早条件已经相同的候选声明之间比较。高优先级不能越过更早阶段。

用三组计数 (A, B, C) 表示：A 是 ID 选择器数量；B 是类、属性选择器与普通伪类数量；C 是类型选择器与伪元素数量。先比 A，相等再比 B，再比 C；它不是十进制分数，许多类也不会向 ID 一栏“进位”。

- ID 选择器 #specific-target 是 (1, 0, 0)。
- .specific-panel p.notice 是 (0, 2, 1)。
- 通配选择器 * 与组合器不增加计数。
- HTML 的 class 排列顺序不决定声明顺序；相同优先级时看 CSS 声明的文档顺序。
- 普通逗号列表分别计算选择器，取实际匹配分支中最高的优先级。

如果这些仍相同，使用 @scope 的规则还要比较作用域接近度，最后才比出现顺序。本章未使用 @scope，具体机制留到第19章。

```html
<div class="specific-panel">
  <p id="specific-target" class="notice">比较 ID 与多个类的选择器</p>
</div>
```

```css
#specific-target { color: navy; }
.specific-panel p.notice { color: maroon; }
/* 本例同来源、同重要性、同层：ID 一栏先决出胜负，后写的 maroon 仍被覆盖。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-3)

## 4 函数伪类的特殊计数

:is()、:not()、:has() 本身不按普通伪类再加一份权重，而是使用参数列表中最高的选择器优先级；这个参数不必就是当前命中的分支。外面的选择器继续累加。

:where() 连同参数始终贡献零。因此 .where-target 的 (0, 1, 0) 能覆盖 :where(#where-target) 的 (0, 0, 0)。这使基础规则容易被后续组件规则覆盖。

下面的 #never-used 没有对应元素，只用来说明 :is() 的计数机制；实际样式不宜为了抬高权重加入虚构 ID。

```html
<p class="is-target">is 参数权重</p>
<p id="where-target" class="where-target">where 零权重</p>
```

```css
:is(.is-target, #never-used) { color: navy; }
.is-target { color: maroon; }
.where-target { color: teal; }
:where(#where-target) { color: maroon; }
/* 检查：第一段深蓝；第二段蓝绿色，尽管 where 规则后写且参数含 ID。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-4)

## 5 继承不是祖先与子元素比权重

继承（inheritance）在元素自身没有相应层叠值时为可继承属性提供默认值，通常取父元素的计算值。color、font-size 等可继承；border、padding 等默认不继承，是否继承须看属性定义。

直接匹配子元素的有效声明与从父元素传来的默认值不是同一组选择器竞争。父元素的 ID 选择器甚至 !important，都不会迫使子元素放弃自己已有的 color；!important 标记也不会随继承传递。

透明背景能透出父背景，但这不意味着 background-color 被继承。继承与视觉透出要分开判断。

```html
<div id="inherit-parent">
  <span id="inherit-auto">使用父文字颜色</span>
  <span class="inherit-own">使用自己的文字颜色</span>
</div>
```

```css
#inherit-parent { color: navy !important; border: 2px solid teal; padding: 12px; }
.inherit-own { color: maroon; }
/* 检查：第一个 span 深蓝且没有自己的边框；第二个 span 为 maroon。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-5)

## 6 initial、inherit 与 unset

这三个 CSS 全局关键字可以作为属性值使用，不是属性或选择器。

- initial 取属性规范的初始值；它不是“恢复这个 HTML 元素的浏览器默认外观”。
- inherit 明确取父元素的计算值，即使该属性默认不继承。
- unset 对可继承属性按 inherit 处理，对不可继承属性按 initial 处理。

本例 padding 的初始值为 0，color 默认继承。对多个属性统一处理时可使用 all 简写，但它不重置 direction、unicode-bidi 和自定义属性；随意使用 all: initial 还会改变 display 等布局条件。这里逐个属性设置，便于观察。

```html
<div class="reset-parent">
  <p class="reset-initial">initial 内边距</p>
  <p class="reset-inherit">inherit 内边距</p>
  <p class="reset-unset">unset 颜色与内边距</p>
</div>
```

```css
.reset-parent { color: navy; padding: 20px; border: 1px solid gray; }
.reset-parent > p { color: maroon; padding: 10px; border: 1px solid teal; }
.reset-parent > .reset-initial { padding: initial; }
.reset-parent > .reset-inherit { padding: inherit; }
.reset-parent > .reset-unset { color: unset; padding: unset; }
/* 检查三个段落的 padding：0、20px、0；最后一段颜色从父元素继承。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-6)

## 7 revert 回退来源

revert 是另一种全局关键字。用于作者样式时，它为当前元素的当前属性回退整个作者来源，再按用户或浏览器来源以及默认处理确定值，不是简单回到 CSS 文件的上一条声明。

initial 直接使用规范初始值；revert 则可能恢复浏览器按元素类型设置的规则。下面 display 的初始值为 inline，而常见浏览器默认把 &lt;div&gt; 显示为 block，两个关键词因而适合对照。

revert 只影响写出该声明的属性和元素，并不自动清除后代自身的规则；如果低来源没有相应声明，可继承属性仍可能通过继承获得父值。

```html
<div class="origin-initial">display: initial</div>
<div class="origin-revert">display: revert</div>
```

```css
.origin-initial, .origin-revert { display: inline-block; border: 1px solid teal; }
.origin-initial { display: initial; }
.origin-revert { display: revert; }
/* 常见默认样式下检查 display：前者 inline，后者 block；用户样式可能影响后者。 */
```

配套文件：[index.html](scripts/03-cascade-and-inheritance/index.html)、[index.css](scripts/03-cascade-and-inheritance/index.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/index.html#demo-7)

## 8 层叠层与重要声明的反转

@layer 是给样式分层的 @ 规则，foundation、theme 是本例自定层名。开头的语句先确定顺序；后面重复打开同名层只是补充规则，不把它移到最后。

同一来源中，普通样式规则按“先声明的层 → 后声明的层 → 未分层规则”递增优先。层顺序先于选择器优先级比较，因此后层的类选择器可以覆盖前层的 ID 选择器。

!important 声明的层顺序反转：前层的重要规则高于后层，所有显式层中的重要规则又高于未分层的重要规则；但任何普通声明都不会因此越过重要声明。内联样式仍按前面说明的附着规则处理。

@layer 的基础语法已在现代主流浏览器实现；旧环境会忽略不认识的整块 @layer。本章保留普通 HTML 内容和未分层的边框，旧环境仍可阅读；要依赖层实现覆盖，应先核对目标版本。不要无意把所有回退颜色写在未分层规则中，否则它们会压过普通层样式。

```html
<p id="layer-specific" class="layer-normal">层顺序先于 ID 优先级</p>
<p class="layer-free">未分层的普通声明</p>
<p class="layer-important">重要声明反转层顺序</p>
```

```css
@layer foundation, theme;
@layer foundation {
  #layer-specific, .layer-free { color: maroon; }
  .layer-important { color: navy !important; }
}
@layer theme {
  .layer-normal, .layer-free { color: teal; }
  .layer-important { color: teal !important; }
}
.layer-normal, .layer-free, .layer-important { border: 1px solid gray; }
.layer-free { color: purple; }
.layer-important { color: maroon !important; }
/* 支持层时依次为 teal、purple、navy；第三段未分层重要声明也输给前层。 */
```

配套文件：[layers.html](scripts/03-cascade-and-inheritance/layers.html)、[layers.css](scripts/03-cascade-and-inheritance/layers.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/layers.html#demo-8)

## 9 revert-layer 回退当前层并追踪声明

revert-layer 的目标是当前层，不是整个作者来源。下面沿用前节 foundation、theme 的顺序；theme 中最后的 revert-layer 获胜后，当前元素的 color 会按移除该层相应声明后重新确定，因此找到 foundation 中的 navy。

它不会回到 theme 中上一条 teal，也不等同 initial。更一般地，回退沿层叠优先关系进行，涉及重要声明时不能只按源码向上找；没有可用的低优先级同来源声明时，才继续回退来源。

在开发者工具中选中 .layer-rollback，过滤 color：先查资源和匹配，再看重要性、所在层、选择器优先级及顺序；取消 revert-layer 后重新观察。Computed 中显示最终呈现相关值，展开来源可追到规则；被划线可能表示覆盖，不等于语法错误。

```html
<p class="layer-rollback">只退回当前层</p>
```

```css
@layer foundation {
  .layer-rollback { color: navy; }
}
@layer theme {
  .layer-rollback { color: teal; }
  .layer-rollback { color: revert-layer; }
}
/* 支持 revert-layer 时为 navy；只取消最后一条声明后变为 teal。 */
```

配套文件：[layers.html](scripts/03-cascade-and-inheritance/layers.html)、[layers.css](scripts/03-cascade-and-inheritance/layers.css) · [浏览器预览](http://127.0.0.1:8101/scripts/03-cascade-and-inheritance/layers.html#demo-9)

## 本章小结

- 先比较来源、重要性等更早阶段，再在同一阶段比较选择器优先级；最后才可能由声明顺序决定。
- 继承提供子元素的默认值，父元素的 !important 不会覆盖子元素自身的普通颜色。
- initial、inherit、unset、revert、revert-layer 分别指定不同的默认或回退过程。
- 普通层顺序与重要层顺序相反，调试时必须同时查看声明所在层。

## 练习

在 scripts/03-cascade-and-inheritance/ 的配套文件中操作，先记录原值，练习后恢复。

（1）交换 .order-demo 两条规则的顺序，检查只有发生冲突的 color 改变；随后给前一条 color 添加 !important，再判断顺序是否仍决定颜色。

（2）将 .reset-unset 的 padding 改为 inherit；检查 padding 从 0 变为父元素的值，而不是自身上一条 10px。

（3）仅交换层顺序语句中的 foundation、theme；检查 .layer-normal 和 .layer-important 的颜色按不同方向变化。

（4）在 layers.html 中取消 color: revert-layer，检查 .layer-rollback 回到 theme 的 teal，并在 Styles 中指出生效文件与声明。

### 提示

每次只改一个条件；先排除更早层叠阶段的差异，再计算 (A, B, C)。

## 参考与引用来源

- W3C：[CSS Cascade Level 5 §6.1–6.4](https://www.w3.org/TR/css-cascade-5/#cascade-sort) 的来源、重要性、元素附着样式和层顺序；[§7.1–7.3.5](https://www.w3.org/TR/css-cascade-5/#defaulting) 的初始值、继承与五种全局关键字；[Selectors Level 4 §15](https://www.w3.org/TR/selectors-4/#specificity-rules) 的计数、列表和函数伪类特例。
- MDN：[Introduction to the CSS cascade](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Cascade/Introduction#cascading_order) 的分阶段顺序和作用域接近度；[Specificity](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Cascade/Specificity#how_is_specificity_calculated) 的比较前提；[Inheritance](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Cascade/Inheritance) 的继承与非继承属性；[revert](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/revert) 的来源回退与 display 对照；[@layer](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@layer#description) 的层顺序、重要性反转和兼容性；[revert-layer](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Values/revert-layer#revert-layer_vs._revert) 的层与来源区别；[all](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/all) 的重置例外。
- Chrome for Developers：[CSS features reference](https://developer.chrome.com/docs/devtools/css/reference#filter) 的 Styles/Computed 属性过滤与规则查看。